# Space-Grade Fault Detector — LibreLane Classic Flow
Run from `/Desktop/Space-Grade-Mechanical-Fault-Detector/` so relative paths resolve.

## 1. Environment check

In [ ]:
import os
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("CWD:", os.getcwd())
print("rtl files:", os.listdir("rtl"))

## 2. Confirm top.v port list before setting CLOCK_PORT
Don't guess the clock/reset names — print them and read off the actual ports.

In [ ]:
with open("rtl/top.v") as f:
    print(f.read()[:2000])  # inspect the module port declaration at the top

## 3. TMR keep-attributes — verify before synth, don't skip
Confirm `(* keep *)` / `dont_touch` is present on the redundant register copies in
`tmr_reg_bank.v`, `axis_sequencer.v`, `goertzel_core.v`, `magnitude_compute.v`
before running step 4. If missing, Yosys will merge the triplicated flops.

## 4. LibreLane Classic flow — harden `top` as a macro

In [ ]:
from librelane.flows import Flow

Classic = Flow.factory.get("Classic")

flow = Classic(
    {
        "PDK": "gf180mcuD",
        "DESIGN_NAME": "top",
        "VERILOG_FILES": ["dir::rtl/*.v"],
        "CLOCK_PORT": "REPLACE_WITH_ACTUAL_PORT",   # from cell 2 output
        "CLOCK_PERIOD": 20,                          # ns, adjust to your target freq
    },
    design_dir=".",
)

flow.start()

## 5. Post-synthesis check — confirm TMR flop count survived
Inspect the Yosys stats / netlist for `tmr_reg_bank` before proceeding to PnR.

In [ ]:
# Point this at the actual run directory LibreLane just created
# e.g. runs/<timestamp>/reports/synthesis/1-synthesis.stat.rpt
runs_dir = sorted(os.listdir("runs"))[-1]
print(runs_dir)

## 6. Next stage (separate notebook cell block, later)
Chip flow for pad ring / top-level assembly — only after step 5 confirms a clean, TMR-intact macro.
```python
Chip = Flow.factory.get("Chip")
```